In [1]:
# ============================================================
# 0. Imports
# ============================================================

from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from difflib import SequenceMatcher

import hashlib
import json
import platform
import re
import sys
import unicodedata

import pandas as pd

In [2]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D10"
DOCUMENT_NAME = "IMPI — Inquérito Mensal à Produção Industrial"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
INPUT_REPRESENTATION = (
    "Complete deterministically normalised layout-aware structural Markdown"
)

EXPECTED_SOURCE_SHA256 = (
    "fb72aac548f61578bc9ba52448793b81bf61e37eff146f9519124d0794a4f4e5"
)

EXPECTED_RECORD_COUNT = 69

EXPECTED_CATEGORY_COUNTS = {
    "Instrument metadata": 8,
    "Questionnaire field": 32,
    "UAE template element": 6,
    "Product table field": 12,
    "Instruction": 11,
}

FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Code",
    "Expected Value Type",
    "Source Location",
]

MANDATORY_STRING_FIELDS = [
    "Category",
    "Section",
    "Field or Concept",
    "Description",
    "Expected Value Type",
    "Source Location",
]

NULLABLE_STRING_FIELDS = ["Code"]

# Frozen from final D10 Branch A validation.
BASE_IDENTITY_FIELDS = [
    "Category",
    "Field or Concept",
]

DUPLICATE_DISAMBIGUATION_FIELD = "Section"

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Field or Concept",
    "Code",
    "Expected Value Type",
    "Source Location",
]

DIAGNOSTIC_FIELDS = [
    "Section",
    "Description",
]

OUTPUT_DIR = Path("outputs_D10_validation_C_revised")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)
print("Output directory:", OUTPUT_DIR)

Document: D10
Branch: C
Expected records: 69
Primary correctness fields: ['Category', 'Field or Concept', 'Code', 'Expected Value Type', 'Source Location']
Output directory: outputs_D10_validation_C_revised


In [3]:
# ============================================================
# 2. Upload canonical Stage 1 + Branch C validation inputs
# ============================================================
# Required:
#   1) D10_reference_values.csv
#   2) D10_branch_C_structure_check.json
#   3) D10_branch_C_experiment_metadata.json
#   4) D10_branch_C_normalisation_check.json
#   5) D10_branch_C_experiment_summary.json
#
# Required only when the Branch C run is content-evaluable:
#   6) D10_branch_C_parsed_extraction.json
#
# The experiment summary is read first so the notebook can verify
# whether a parsed extraction should exist.

print(
    "Upload:\n"
    "1. D10_reference_values.csv\n"
    "2. D10_branch_C_structure_check.json\n"
    "3. D10_branch_C_experiment_metadata.json\n"
    "4. D10_branch_C_normalisation_check.json\n"
    "5. D10_branch_C_experiment_summary.json\n"
    "6. D10_branch_C_parsed_extraction.json if it was created"
)

uploaded = files.upload()
uploaded_paths = [Path(name) for name in uploaded]

csv_paths = [p for p in uploaded_paths if p.suffix.lower() == ".csv"]
json_paths = [p for p in uploaded_paths if p.suffix.lower() == ".json"]

if len(csv_paths) != 1:
    raise ValueError(
        "Upload exactly one CSV file: D10_reference_values.csv."
    )

REFERENCE_PATH = csv_paths[0]

EXTRACTION_PATH = None
STRUCTURE_PATH = None
METADATA_PATH = None
NORMALISATION_INTEGRITY_PATH = None
EXPERIMENT_SUMMARY_PATH = None


def canonical_filename(path):
    return path.name.casefold().replace(" ", "_")


# First pass: filename patterns.
for path in json_paths:
    filename = canonical_filename(path)

    if "d10_branch_c_parsed_extraction" in filename:
        EXTRACTION_PATH = path
        continue

    if "d10_branch_c_structure_check" in filename:
        STRUCTURE_PATH = path
        continue

    if (
        "d10_branch_c_experiment_metadata" in filename
        and "_pre" not in filename
    ):
        METADATA_PATH = path
        continue

    if (
        "d10_branch_c_normalisation_check" in filename
        or "d10_branch_c_normalization_check" in filename
    ):
        NORMALISATION_INTEGRITY_PATH = path
        continue

    if "d10_branch_c_experiment_summary" in filename:
        EXPERIMENT_SUMMARY_PATH = path
        continue


# Second pass: content-based fallback.
for path in json_paths:
    with path.open("r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        EXPERIMENT_SUMMARY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "parsed_extraction_created" in obj
        and "validation_status" in obj
    ):
        EXPERIMENT_SUMMARY_PATH = path
        continue

    if (
        NORMALISATION_INTEGRITY_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and obj.get("parent_branch") == "B"
        and "normalisation_integrity_passed" in obj
        and "parent_equivalence_passed" in obj
    ):
        NORMALISATION_INTEGRITY_PATH = path
        continue

    if (
        METADATA_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "source_sha256" in obj
        and "raw_response_sha256" in obj
        and "structure_check_file" in obj
    ):
        METADATA_PATH = path
        continue

    if (
        STRUCTURE_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structure_valid" in obj
        and "records_evaluable" in obj
        and "validation_status" not in obj
    ):
        STRUCTURE_PATH = path
        continue

    if (
        EXTRACTION_PATH is None
        and obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        EXTRACTION_PATH = path
        continue


for label, path in {
    "structure check": STRUCTURE_PATH,
    "experiment metadata": METADATA_PATH,
    "normalisation check": NORMALISATION_INTEGRITY_PATH,
    "experiment summary": EXPERIMENT_SUMMARY_PATH,
}.items():
    if path is None:
        raise ValueError(f"Could not identify required {label} file.")


experiment_summary = json.loads(
    EXPERIMENT_SUMMARY_PATH.read_text(encoding="utf-8")
)

content_evaluable = bool(
    experiment_summary.get("records_evaluable")
    and experiment_summary.get("parsed_extraction_created")
)

if content_evaluable and EXTRACTION_PATH is None:
    raise ValueError(
        "This Branch C execution is content-evaluable, so "
        "D10_branch_C_parsed_extraction.json is required."
    )

if not content_evaluable:
    raise ValueError(
        "This D10 Branch C run is not content-evaluable. "
        "Do not use the standard content-validation pathway. "
        "Send the experiment summary so a non-evaluable validation "
        "notebook can be used, as was done for D9."
    )


print("\nIdentified D10 Branch C validation inputs:")
print("Reference:", REFERENCE_PATH.name)
print("Parsed extraction:", EXTRACTION_PATH.name)
print("Structure check:", STRUCTURE_PATH.name)
print("Experiment metadata:", METADATA_PATH.name)
print("Normalisation check:", NORMALISATION_INTEGRITY_PATH.name)
print("Experiment summary:", EXPERIMENT_SUMMARY_PATH.name)
print("Content evaluable:", content_evaluable)

Upload:
1. D10_reference_values.csv
2. D10_branch_C_structure_check.json
3. D10_branch_C_experiment_metadata.json
4. D10_branch_C_normalisation_check.json
5. D10_branch_C_experiment_summary.json
6. D10_branch_C_parsed_extraction.json if it was created


Saving D10_branch_C_structure_check.json to D10_branch_C_structure_check.json
Saving D10_branch_C_parsed_extraction.json to D10_branch_C_parsed_extraction.json
Saving D10_branch_C_normalisation_check.json to D10_branch_C_normalisation_check.json
Saving D10_branch_C_experiment_summary.json to D10_branch_C_experiment_summary.json
Saving D10_branch_C_experiment_metadata.json to D10_branch_C_experiment_metadata.json
Saving D10_reference_values.csv to D10_reference_values.csv

Identified D10 Branch C validation inputs:
Reference: D10_reference_values.csv
Parsed extraction: D10_branch_C_parsed_extraction.json
Structure check: D10_branch_C_structure_check.json
Experiment metadata: D10_branch_C_experiment_metadata.json
Normalisation check: D10_branch_C_normalisation_check.json
Experiment summary: D10_branch_C_experiment_summary.json
Content evaluable: True


In [4]:
# ============================================================
# 3. Load inputs and verify provenance
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
STRUCTURE_SHA256 = sha256_file(STRUCTURE_PATH)
METADATA_SHA256 = sha256_file(METADATA_PATH)
NORMALISATION_INTEGRITY_SHA256 = sha256_file(
    NORMALISATION_INTEGRITY_PATH
)
EXPERIMENT_SUMMARY_SHA256 = sha256_file(
    EXPERIMENT_SUMMARY_PATH
)


reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=True,
)

reference_df = reference_df.where(
    pd.notna(reference_df),
    None,
)


parsed_extraction = json.loads(
    EXTRACTION_PATH.read_text(encoding="utf-8")
)

if not isinstance(parsed_extraction, dict):
    raise ValueError(
        "Canonical parsed extraction must be a top-level JSON object."
    )

document_id_correct = (
    parsed_extraction.get("document_id") == DOCUMENT_ID
)

branch_correct = (
    parsed_extraction.get("branch") == BRANCH
)

extracted_records = parsed_extraction.get("records")
records_is_list = isinstance(extracted_records, list)

if not records_is_list:
    raise ValueError(
        "Parsed Branch C extraction must contain a records list."
    )

extracted_df = pd.DataFrame(extracted_records)


structure_check = json.loads(
    STRUCTURE_PATH.read_text(encoding="utf-8")
)

experiment_metadata = json.loads(
    METADATA_PATH.read_text(encoding="utf-8")
)

normalisation_integrity = json.loads(
    NORMALISATION_INTEGRITY_PATH.read_text(encoding="utf-8")
)


for artefact_name, artefact in {
    "structure check": structure_check,
    "experiment metadata": experiment_metadata,
    "normalisation integrity": normalisation_integrity,
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id: "
            f"{artefact.get('document_id')}"
        )

    if artefact.get("branch") != BRANCH:
        raise ValueError(
            f"Unexpected {artefact_name} branch: "
            f"{artefact.get('branch')}"
        )


branch_c_structure_valid = bool(
    structure_check.get("structure_valid")
)

parsed_extraction_hash_matches_metadata = (
    experiment_metadata.get("parsed_extraction_sha256")
    == EXTRACTION_SHA256
)

source_hash_matches_stage_1 = (
    experiment_metadata.get("source_sha256")
    == EXPECTED_SOURCE_SHA256
)

parent_equivalence_passed = bool(
    normalisation_integrity.get(
        "parent_equivalence_passed",
        False,
    )
)

normalisation_integrity_passed = bool(
    normalisation_integrity.get(
        "normalisation_integrity_passed",
        False,
    )
)


print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print("Branch C structure valid:", branch_c_structure_valid)
print(
    "Parsed extraction hash matches metadata:",
    parsed_extraction_hash_matches_metadata,
)
print(
    "Source hash matches Stage 1:",
    source_hash_matches_stage_1,
)
print(
    "Parent B equivalence passed:",
    parent_equivalence_passed,
)
print(
    "Normalisation integrity passed:",
    normalisation_integrity_passed,
)


if not parsed_extraction_hash_matches_metadata:
    raise AssertionError(
        "The parsed extraction does not match Branch C experiment metadata."
    )

if not source_hash_matches_stage_1:
    raise AssertionError(
        "Branch C did not use the fixed Stage 1 D10 source."
    )

if not parent_equivalence_passed:
    raise AssertionError(
        "Branch C parent-B equivalence did not pass."
    )

if not normalisation_integrity_passed:
    raise AssertionError(
        "Branch C normalisation integrity did not pass."
    )

Reference SHA-256: 4dba65cc0dc751c980edc61ca68d428f780c9bedf9db82be3ab0d04cf02d0734
Extraction SHA-256: 317ede744eb06fb3ebe1f2e2e2bfc732d07e6a2b27bafe6b8189895c55d28a46
Branch C structure valid: True
Parsed extraction hash matches metadata: True
Source hash matches Stage 1: True
Parent B equivalence passed: True
Normalisation integrity passed: True


In [5]:
# ============================================================
# 5. Schema, type and content diagnostics
# ============================================================

reference_schema_exact = (
    reference_df.columns.tolist() == FIELDS
)

extraction_schema_exact = (
    extracted_df.columns.tolist() == FIELDS
)


def validate_record_types(records, dataset_name):
    issues = []

    for index, record in enumerate(records):

        if not isinstance(record, dict):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": None,
                "issue": "Record is not an object",
            })
            continue

        observed_fields = list(record.keys())

        # Field order is a diagnostic only.
        if set(observed_fields) != set(FIELDS):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": None,
                "issue": "Field set differs from schema",
                "observed_fields": observed_fields,
            })

        for field in MANDATORY_STRING_FIELDS:
            value = record.get(field)

            if value is None or (
                isinstance(value, str)
                and not value.strip()
            ):
                issues.append({
                    "dataset": dataset_name,
                    "record_index": index,
                    "field": field,
                    "issue": "Missing mandatory value",
                })

            elif not isinstance(value, str):
                issues.append({
                    "dataset": dataset_name,
                    "record_index": index,
                    "field": field,
                    "issue": "Expected string",
                    "observed_type": type(value).__name__,
                })

        code_value = record.get("Code")

        if (
            code_value is not None
            and not isinstance(code_value, str)
        ):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": "Code",
                "issue": "Expected string or null",
                "observed_type": type(code_value).__name__,
            })

    return issues


reference_type_issues = validate_record_types(
    reference_df.to_dict("records"),
    "Reference",
)

extraction_type_issues = validate_record_types(
    extracted_records,
    "Extraction",
)

reference_types_valid = (
    len(reference_type_issues) == 0
)

extraction_types_valid = (
    len(extraction_type_issues) == 0
)


reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

extraction_record_count_valid = (
    len(extracted_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

extraction_category_counts = (
    extracted_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

extraction_category_counts_valid = (
    extraction_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


schema_validity = all([
    document_id_correct,
    branch_correct,
    records_is_list,
    extraction_schema_exact,
    extraction_types_valid,
])


print("Reference schema exact:", reference_schema_exact)
print("Extraction schema exact:", extraction_schema_exact)
print("Reference types valid:", reference_types_valid)
print("Extraction types valid:", extraction_types_valid)
print("Schema validity:", schema_validity)
print("Reference record count:", len(reference_df))
print("Extraction record count:", len(extracted_df))
print("Reference category counts:", reference_category_counts)
print("Extraction category counts:", extraction_category_counts)


Reference schema exact: True
Extraction schema exact: True
Reference types valid: True
Extraction types valid: True
Schema validity: True
Reference record count: 69
Extraction record count: 69
Reference category counts: {'Questionnaire field': 32, 'Product table field': 12, 'Instruction': 11, 'Instrument metadata': 8, 'UAE template element': 6}
Extraction category counts: {'Questionnaire field': 32, 'Product table field': 12, 'Instruction': 11, 'Instrument metadata': 8, 'UAE template element': 6}


In [6]:
# ============================================================
# 6. Confirm current Stage 1 D10 reference semantics
# ============================================================

EXPECTED_UAE_TEMPLATE_LABELS = {
    "Código da UAE",
    "Designação da UAE",
    "Situação da UAE perante a atividade",
    "Observações da UAE",
    "Confirmar",
    "Produtos",
}

EXPECTED_PRODUCT_TABLE_LABELS = {
    "NIF",
    "UAE",
    "Período de Referência",
    "Nº",
    "Produto",
    "Unid.",
    "Código",
    "Quantidades produzidas",
    "Quantidades vendidas",
    "Valor das vendas / prestação de serviços",
    "Observações empresa",
    "Observações INE",
}


observed_uae_template_labels = set(
    reference_df.loc[
        reference_df["Category"]
        == "UAE template element",
        "Field or Concept",
    ]
)

observed_product_table_labels = set(
    reference_df.loc[
        reference_df["Category"]
        == "Product table field",
        "Field or Concept",
    ]
)


reference_period_field_valid = (
    (
        reference_df["Field or Concept"]
        == "Referência dos dados"
    ).sum()
    == 1
)

source_location_pattern_valid = bool(
    reference_df["Source Location"]
    .fillna("")
    .str.match(r"^PDF page [1-4] — .+$")
    .all()
)


reference_semantic_checks = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "reference_schema_exact":
        bool(reference_schema_exact),

    "reference_types_valid":
        bool(reference_types_valid),

    "uae_template_valid":
        observed_uae_template_labels
        == EXPECTED_UAE_TEMPLATE_LABELS,

    "reference_period_field_valid":
        bool(reference_period_field_valid),

    "product_table_structure_valid":
        observed_product_table_labels
        == EXPECTED_PRODUCT_TABLE_LABELS,

    "source_location_pattern_valid":
        bool(source_location_pattern_valid),
}


reference_semantics_valid = all(
    reference_semantic_checks.values()
)

print(
    json.dumps(
        reference_semantic_checks,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "Corrected/current D10 reference semantics valid:",
    reference_semantics_valid,
)

if not reference_semantics_valid:
    raise AssertionError(
        "The supplied D10 reference does not match the current "
        "frozen Stage 1 D10 reference semantics."
    )


{
  "reference_record_count_valid": true,
  "reference_category_counts_valid": true,
  "reference_schema_exact": true,
  "reference_types_valid": true,
  "uae_template_valid": true,
  "reference_period_field_valid": true,
  "product_table_structure_valid": true,
  "source_location_pattern_valid": true
}
Corrected/current D10 reference semantics valid: True


In [7]:
# ============================================================
# 7. Comparison-only normalisation
# ============================================================

def normalise_text(value):
    if value is None:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = (
        text
        .replace("—", "-")
        .replace("–", "-")
        .replace("‑", "-")
        .replace("“", '"')
        .replace("”", '"')
        .replace("’", "'")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text.casefold()


def normalise_identity_text(value):
    text = normalise_text(value)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


SECTION_EQUIVALENCE_GROUPS = [
    {
        "header",
        "questionnaire header",
        "legal notice",
    },
    {
        "response contacts",
        "contact and response information",
        "response information",
    },
    {
        "reference data",
        "reference-data area",
    },
    {
        "identification of statistical unit",
        "section i",
        "i - identificação da unidade estatística",
        "identificação da unidade estatística",
    },
    {
        "activity status",
        "section ii",
        "ii - situação da unidade estatística no período de referência dos dados",
        "situação da unidade estatística no período de referência dos dados",
    },
    {
        "observations",
        "section iii",
        "iii - observações",
        "observações",
    },
    {
        "responsible person",
        "section iv",
        "iv - responsável pelo preenchimento",
        "responsável pelo preenchimento",
    },
    {
        "uae information",
        "uae template",
    },
    {
        "product production table",
        "product table",
        "product table header",
        "product table columns",
    },
    {
        "filling instructions",
        "instruções de preenchimento",
    },
    {
        "explanatory notes",
        "notas explicativas",
    },
]


def canonical_section(value):
    value_norm = normalise_identity_text(value)

    for group_index, group in enumerate(
        SECTION_EQUIVALENCE_GROUPS
    ):
        normalised_group = {
            normalise_identity_text(item)
            for item in group
        }

        if value_norm in normalised_group:
            return f"section_group_{group_index}"

    return value_norm


def normalise_code(value):
    if value is None:
        return None

    text = normalise_text(value)

    if not text:
        return None

    return text.upper()


def extract_pdf_page(value):
    text = normalise_text(value)

    match = re.search(
        r"(?:physical\s+)?pdf\s+page\s+(\d+)",
        text,
    )

    if not match:
        return None

    return int(match.group(1))


def description_similarity(a, b):
    a_norm = normalise_text(a)
    b_norm = normalise_text(b)

    if not a_norm and not b_norm:
        return 1.0

    if not a_norm or not b_norm:
        return 0.0

    return SequenceMatcher(
        None,
        a_norm,
        b_norm,
    ).ratio()


In [8]:
# ============================================================
# 8. Deterministic one-to-one identity alignment
# ============================================================

# Base identity = Category + Field or Concept.
# Section is used only for repeated labels within the same category.


def base_identity(record):
    return (
        normalise_identity_text(
            record["Category"]
        ),
        normalise_identity_text(
            record["Field or Concept"]
        ),
    )


def section_identity(record):
    return canonical_section(
        record["Section"]
    )


reference_records = (
    reference_df
    .to_dict("records")
)

extraction_records = (
    extracted_df
    .to_dict("records")
)


reference_groups = defaultdict(list)
extraction_groups = defaultdict(list)

for index, record in enumerate(reference_records):
    reference_groups[
        base_identity(record)
    ].append(index)

for index, record in enumerate(extraction_records):
    extraction_groups[
        base_identity(record)
    ].append(index)


matches = []
matched_reference = set()
matched_extraction = set()
ambiguous_alignment_groups = []


all_base_keys = sorted(
    set(reference_groups)
    | set(extraction_groups)
)


for key in all_base_keys:

    ref_indices = reference_groups.get(
        key,
        [],
    )

    ext_indices = extraction_groups.get(
        key,
        [],
    )

    # Unique base identity: direct deterministic match.
    if (
        len(ref_indices) == 1
        and len(ext_indices) == 1
    ):
        r_idx = ref_indices[0]
        e_idx = ext_indices[0]

        matches.append({
            "Reference Index": r_idx,
            "Extraction Index": e_idx,
            "Alignment Rule":
                "Category + Field or Concept",
        })

        matched_reference.add(r_idx)
        matched_extraction.add(e_idx)
        continue


    # Repeated label: disambiguate only by canonical Section.
    ref_by_section = defaultdict(list)
    ext_by_section = defaultdict(list)

    for r_idx in ref_indices:
        ref_by_section[
            section_identity(
                reference_records[r_idx]
            )
        ].append(r_idx)

    for e_idx in ext_indices:
        ext_by_section[
            section_identity(
                extraction_records[e_idx]
            )
        ].append(e_idx)


    for section_key in sorted(
        set(ref_by_section)
        | set(ext_by_section)
    ):

        r_list = ref_by_section.get(
            section_key,
            [],
        )

        e_list = ext_by_section.get(
            section_key,
            [],
        )

        if (
            len(r_list) == 1
            and len(e_list) == 1
        ):
            r_idx = r_list[0]
            e_idx = e_list[0]

            matches.append({
                "Reference Index": r_idx,
                "Extraction Index": e_idx,
                "Alignment Rule":
                    "Category + Field or Concept + canonical Section",
            })

            matched_reference.add(r_idx)
            matched_extraction.add(e_idx)

        elif r_list or e_list:
            ambiguous_alignment_groups.append({
                "base_identity": key,
                "canonical_section": section_key,
                "reference_indices": r_list,
                "extraction_indices": e_list,
            })


missing_reference_indices = sorted(
    set(range(len(reference_records)))
    - matched_reference
)

unsupported_extraction_indices = sorted(
    set(range(len(extraction_records)))
    - matched_extraction
)


print("Aligned:", len(matches))
print("Missing:", len(missing_reference_indices))
print(
    "Unsupported/unmatched:",
    len(unsupported_extraction_indices),
)
print(
    "Ambiguous identity groups:",
    len(ambiguous_alignment_groups),
)


Aligned: 69
Missing: 0
Unsupported/unmatched: 0
Ambiguous identity groups: 0


In [9]:
# ============================================================
# 9. Field comparison
# ============================================================

def exact_text_equal(a, b):
    return (
        normalise_text(a)
        == normalise_text(b)
    )


def section_equal(a, b):
    return (
        canonical_section(a)
        == canonical_section(b)
    )


def code_equal(a, b):
    return (
        normalise_code(a)
        == normalise_code(b)
    )


def source_location_equal(a, b):
    # The prompt requires a physical PDF page plus a concise source
    # region. The region wording is intentionally flexible.
    # Primary correctness therefore evaluates grounded physical page.
    page_a = extract_pdf_page(a)
    page_b = extract_pdf_page(b)

    return (
        page_a is not None
        and page_b is not None
        and page_a == page_b
    )


comparison_rows = []


for match in matches:

    r = reference_records[
        match["Reference Index"]
    ]

    e = extraction_records[
        match["Extraction Index"]
    ]

    out = {
        "Reference Index":
            match["Reference Index"],

        "Extraction Index":
            match["Extraction Index"],

        "Alignment Rule":
            match["Alignment Rule"],
    }


    correctness = {
        "Category":
            exact_text_equal(
                r["Category"],
                e["Category"],
            ),

        "Section":
            section_equal(
                r["Section"],
                e["Section"],
            ),

        "Field or Concept":
            exact_text_equal(
                r["Field or Concept"],
                e["Field or Concept"],
            ),

        "Description":
            exact_text_equal(
                r["Description"],
                e["Description"],
            ),

        "Code":
            code_equal(
                r["Code"],
                e["Code"],
            ),

        "Expected Value Type":
            exact_text_equal(
                r["Expected Value Type"],
                e["Expected Value Type"],
            ),

        "Source Location":
            source_location_equal(
                r["Source Location"],
                e["Source Location"],
            ),
    }


    for field in FIELDS:
        out[
            f"Reference {field}"
        ] = r[field]

        out[
            f"Extracted {field}"
        ] = e[field]

        out[
            f"{field} Correct"
        ] = bool(
            correctness[field]
        )


    out[
        "Description Similarity Diagnostic"
    ] = float(
        description_similarity(
            r["Description"],
            e["Description"],
        )
    )


    out[
        "Fully Correct Primary Record"
    ] = all(
        correctness[field]
        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )


    comparison_rows.append(out)


comparison_df = pd.DataFrame(
    comparison_rows
)


missing_records_df = (
    reference_df
    .iloc[
        missing_reference_indices
    ]
    .copy()
)


unsupported_records_df = (
    extracted_df
    .iloc[
        unsupported_extraction_indices
    ]
    .copy()
)


discrepant_records_df = (
    comparison_df.loc[
        ~comparison_df[
            "Fully Correct Primary Record"
        ]
    ]
    .copy()
)


print(
    "Fully correct primary records:",
    int(
        comparison_df[
            "Fully Correct Primary Record"
        ].sum()
    )
)

print(
    "Primary discrepant records:",
    len(discrepant_records_df),
)


Fully correct primary records: 65
Primary discrepant records: 4


In [10]:
# ============================================================
# 10. Metrics
# ============================================================

aligned_records = len(
    comparison_df
)

fully_correct_records = int(
    comparison_df[
        "Fully Correct Primary Record"
    ].sum()
)

discrepant_records = (
    aligned_records
    - fully_correct_records
)

missing_records = len(
    missing_reference_indices
)

unsupported_records = len(
    unsupported_extraction_indices
)


completeness = (
    aligned_records
    / len(reference_df)
    if len(reference_df)
    else 0.0
)

record_precision_exact = (
    fully_correct_records
    / len(extracted_df)
    if len(extracted_df)
    else 0.0
)

record_recall_exact = (
    fully_correct_records
    / len(reference_df)
    if len(reference_df)
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)


primary_field_accuracy = {}

for field in PRIMARY_CORRECTNESS_FIELDS:

    primary_field_accuracy[field] = float(
        comparison_df[
            f"{field} Correct"
        ].mean()
    ) if aligned_records else 0.0


diagnostic_field_accuracy = {
    "Section": float(
        comparison_df[
            "Section Correct"
        ].mean()
    ) if aligned_records else 0.0,

    "Description exact": float(
        comparison_df[
            "Description Correct"
        ].mean()
    ) if aligned_records else 0.0,

    "Description mean lexical similarity": float(
        comparison_df[
            "Description Similarity Diagnostic"
        ].mean()
    ) if aligned_records else 0.0,
}


overall_primary_field_accuracy = (
    sum(
        primary_field_accuracy.values()
    )
    / len(
        primary_field_accuracy
    )
)


category_metrics = {}

for category, expected in (
    EXPECTED_CATEGORY_COUNTS.items()
):

    ref_subset = reference_df[
        reference_df["Category"]
        == category
    ]

    ext_subset = extracted_df[
        extracted_df["Category"]
        == category
    ]

    aligned_subset = comparison_df[
        comparison_df[
            "Reference Category"
        ]
        == category
    ]

    full = int(
        aligned_subset[
            "Fully Correct Primary Record"
        ].sum()
    )

    category_metrics[category] = {
        "expected_records": int(expected),
        "extracted_records": int(
            len(ext_subset)
        ),
        "aligned_records": int(
            len(aligned_subset)
        ),
        "fully_correct_records": full,
        "discrepant_records": int(
            len(aligned_subset) - full
        ),
        "completeness": (
            len(aligned_subset)
            / expected
            if expected
            else 0.0
        ),
        "record_precision_exact": (
            full
            / len(ext_subset)
            if len(ext_subset)
            else 0.0
        ),
        "record_recall_exact": (
            full
            / len(ref_subset)
            if len(ref_subset)
            else 0.0
        ),
    }

    p = category_metrics[
        category
    ][
        "record_precision_exact"
    ]

    r = category_metrics[
        category
    ][
        "record_recall_exact"
    ]

    category_metrics[
        category
    ][
        "record_f1_exact"
    ] = (
        2 * p * r / (p + r)
        if p + r
        else 0.0
    )


print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_df))
print("Aligned records:", aligned_records)
print("Fully correct records:", fully_correct_records)
print("Discrepant records:", discrepant_records)
print("Missing records:", missing_records)
print("Unsupported/unmatched records:", unsupported_records)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Overall primary field accuracy:",
    round(overall_primary_field_accuracy, 4),
)
print("Schema valid:", schema_validity)


Reference records: 69
Extracted records: 69
Aligned records: 69
Fully correct records: 65
Discrepant records: 4
Missing records: 0
Unsupported/unmatched records: 0
Completeness: 1.0
Exact F1: 0.942
Overall primary field accuracy: 0.9884
Schema valid: True


In [11]:
# ============================================================
# 10. Preserve Branch C normalisation-integrity diagnostics
# ============================================================
# These diagnostics describe the Stage 2 B→C transformation.
# They do NOT determine Stage 4 record correctness.

representation_integrity = {
    "parent_branch":
        normalisation_integrity.get("parent_branch"),

    "parent_equivalence_passed":
        bool(
            normalisation_integrity.get(
                "parent_equivalence_passed",
                False,
            )
        ),

    "normalisation_integrity_passed":
        bool(
            normalisation_integrity.get(
                "normalisation_integrity_passed",
                False,
            )
        ),

    "page_sequence_preserved":
        normalisation_integrity.get("page_sequence_preserved"),

    "deterministic_representation_verified":
        normalisation_integrity.get(
            "deterministic_representation_verified"
        ),

    "all_critical_markers_preserved":
        normalisation_integrity.get(
            "all_critical_markers_preserved"
        ),

    "questionnaire_codes_preserved":
        normalisation_integrity.get(
            "questionnaire_codes_preserved"
        ),

    "sample_product_rows_preserved_in_representation":
        normalisation_integrity.get(
            "sample_product_rows_preserved_in_representation"
        ),

    "repeated_uae_content_preserved":
        normalisation_integrity.get(
            "repeated_uae_content_preserved"
        ),

    "tokens_preserved":
        normalisation_integrity.get("tokens_preserved"),

    "token_preservation":
        normalisation_integrity.get("token_preservation"),

    "complete_4_page_representation_retained":
        normalisation_integrity.get(
            "complete_4_page_representation_retained"
        ),

    "source_scope_filtering_applied":
        normalisation_integrity.get(
            "source_scope_filtering_applied"
        ),

    "page_removal_applied":
        normalisation_integrity.get("page_removal_applied"),

    "page_cropping_applied":
        normalisation_integrity.get("page_cropping_applied"),

    "branch_B_structural_conversion_inherited":
        normalisation_integrity.get(
            "branch_B_structural_conversion_inherited"
        ),

    "branch_B_regeneration_attempted":
        normalisation_integrity.get(
            "branch_B_regeneration_attempted"
        ),

    "ocr_applied":
        normalisation_integrity.get("ocr_applied"),

    "unicode_nfkc_normalisation_applied":
        normalisation_integrity.get(
            "unicode_nfkc_normalisation_applied"
        ),

    "unicode_space_standardisation_applied":
        normalisation_integrity.get(
            "unicode_space_standardisation_applied"
        ),

    "apostrophe_standardisation_applied":
        normalisation_integrity.get(
            "apostrophe_standardisation_applied"
        ),

    "dash_and_minus_standardisation_applied":
        normalisation_integrity.get(
            "dash_and_minus_standardisation_applied"
        ),

    "soft_hyphen_removal_applied":
        normalisation_integrity.get(
            "soft_hyphen_removal_applied"
        ),

    "line_endings_standardised":
        normalisation_integrity.get(
            "line_endings_standardised"
        ),

    "horizontal_whitespace_normalisation_applied":
        normalisation_integrity.get(
            "horizontal_whitespace_normalisation_applied"
        ),

    "semantic_harmonisation_applied":
        normalisation_integrity.get(
            "semantic_harmonisation_applied"
        ),

    "semantic_rewriting_applied":
        normalisation_integrity.get(
            "semantic_rewriting_applied"
        ),

    "questionnaire_code_rewriting_applied":
        normalisation_integrity.get(
            "questionnaire_code_rewriting_applied"
        ),

    "source_spelling_correction_applied":
        normalisation_integrity.get(
            "source_spelling_correction_applied"
        ),

    "unit_conversion_applied":
        normalisation_integrity.get(
            "unit_conversion_applied"
        ),

    "numeric_calculation_applied":
        normalisation_integrity.get(
            "numeric_calculation_applied"
        ),

    "manual_reconstruction_applied":
        normalisation_integrity.get(
            "manual_reconstruction_applied"
        ),

    "manual_correction_applied":
        normalisation_integrity.get(
            "manual_correction_applied"
        ),

    "reference_values_used_for_transformation":
        normalisation_integrity.get(
            "reference_values_used_for_transformation"
        ),
}

print("Branch C representation integrity:")
print(
    json.dumps(
        representation_integrity,
        ensure_ascii=False,
        indent=2,
    )
)

Branch C representation integrity:
{
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "all_critical_markers_preserved": true,
  "questionnaire_codes_preserved": true,
  "sample_product_rows_preserved_in_representation": true,
  "repeated_uae_content_preserved": true,
  "tokens_preserved": true,
  "token_preservation": {
    "questionnaire_codes": {
      "count_before": 8,
      "count_after": 8,
      "missing_token_count": 0,
      "added_token_count": 0,
      "passed": true
    },
    "dates": {
      "count_before": 1,
      "count_after": 1,
      "missing_token_count": 0,
      "added_token_count": 0,
      "passed": true
    },
    "postal_codes": {
      "count_before": 1,
      "count_after": 1,
      "missing_token_count": 0,
      "added_token_count": 0,
      "passed": true
    },
    "integers_and_decimals": {
      "count_before": 44

In [12]:
# ============================================================
# 11. Build final Branch C validation summary
# ============================================================

schema_validity = all([
    document_id_correct,
    branch_correct,
    records_is_list,
    extraction_schema_exact,
    extraction_types_valid,
    branch_c_structure_valid,
])

schema_diagnostics = {
    "top_level_object_valid":
        isinstance(parsed_extraction, dict),
    "document_id_correct":
        bool(document_id_correct),
    "branch_correct":
        bool(branch_correct),
    "records_is_list":
        bool(records_is_list),
    "record_schema_valid":
        bool(extraction_schema_exact),
    "field_types_valid":
        bool(extraction_types_valid),
    "branch_C_structure_valid":
        bool(branch_c_structure_valid),
    "schema_validity":
        bool(schema_validity),
}

content_diagnostics = {
    "reference_record_count_valid":
        bool(reference_record_count_valid),
    "reference_category_counts_valid":
        bool(reference_category_counts_valid),
    "extraction_record_count_valid":
        bool(extraction_record_count_valid),
    "extraction_category_counts_valid":
        bool(extraction_category_counts_valid),
    "branch_C_scope_complete":
        structure_check.get("scope_complete"),
    "branch_C_content_diagnostics":
        structure_check.get("content_diagnostics"),
    "ambiguous_alignment_group_count":
        int(len(ambiguous_alignment_groups)),
}

VALIDATION_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "input_representation": INPUT_REPRESENTATION,

    "reference_records": int(len(reference_df)),
    "extracted_records": int(len(extracted_df)),
    "aligned_records": int(aligned_records),
    "fully_correct_records": int(fully_correct_records),
    "discrepant_records": int(discrepant_records),
    "missing_records": int(missing_records),
    "unsupported_extracted_records": int(unsupported_records),

    "completeness": float(completeness),
    "missing_rate": float(
        missing_records / len(reference_df)
        if len(reference_df) else 0.0
    ),
    "record_precision_exact":
        float(record_precision_exact),
    "record_recall_exact":
        float(record_recall_exact),
    "record_f1_exact":
        float(record_f1_exact),
    "unsupported_rate": float(
        unsupported_records / len(extracted_df)
        if len(extracted_df) else 0.0
    ),
    "discrepancy_rate_among_aligned": float(
        discrepant_records / aligned_records
        if aligned_records else 0.0
    ),

    "overall_primary_field_accuracy":
        float(overall_primary_field_accuracy),
    "primary_field_accuracy":
        primary_field_accuracy,
    "diagnostic_field_accuracy":
        diagnostic_field_accuracy,

    "schema_validity":
        bool(schema_validity),
    "schema_diagnostics":
        schema_diagnostics,
    "content_diagnostics":
        content_diagnostics,

    "branch_C_representation_integrity":
        representation_integrity,

    "matching_rules": {
        "base_identity_fields":
            BASE_IDENTITY_FIELDS,
        "duplicate_disambiguation_field":
            DUPLICATE_DISAMBIGUATION_FIELD,
        "one_to_one_assignment": (
            "Deterministic Category + Field or Concept identity; "
            "canonical Section used only when the same concept label "
            "occurs more than once within a category."
        ),
        "code_used_for_alignment": False,
        "expected_value_type_used_for_alignment": False,
        "description_used_for_alignment": False,
        "source_location_used_for_alignment": False,
        "equivalence_rules_frozen": True,
    },

    "comparison_rules": {
        "raw_extraction_modified": False,
        "manual_correction_applied": False,
        "comparison_normalisation_scope":
            "Comparison copies only",
        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,
        "section": (
            "Canonical source-grounded section diagnostic; "
            "excluded from primary exact-record correctness."
        ),
        "description": (
            "Normalised exact diagnostic plus lexical-similarity "
            "diagnostic; excluded from primary exact-record correctness."
        ),
        "code": (
            "Exact printed-code agreement after conservative "
            "case/whitespace normalisation; null must remain null."
        ),
        "expected_value_type": (
            "Normalised exact textual agreement; no fuzzy correctness "
            "and no identifier/numeric-identifier equivalence."
        ),
        "source_location": (
            "Correct physical PDF page required; concise region wording "
            "need not be identical."
        ),
        "d10_equivalence_rules_status": (
            "Final D10 Branch A document/schema-level comparison rules "
            "reused unchanged. No Branch-C-specific performance-driven "
            "equivalence rules were added."
        ),
    },

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(reference_semantics_valid),
        "checks":
            reference_semantic_checks,
        "reference_modified_by_validation":
            False,
    },

    "category_metrics":
        category_metrics,

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,
        "reference_sha256":
            REFERENCE_SHA256,
        "parsed_extraction_file":
            EXTRACTION_PATH.name,
        "parsed_extraction_sha256":
            EXTRACTION_SHA256,
        "structure_check_file":
            STRUCTURE_PATH.name,
        "structure_check_sha256":
            STRUCTURE_SHA256,
        "experiment_metadata_file":
            METADATA_PATH.name,
        "experiment_metadata_sha256":
            METADATA_SHA256,
        "normalisation_integrity_file":
            NORMALISATION_INTEGRITY_PATH.name,
        "normalisation_integrity_sha256":
            NORMALISATION_INTEGRITY_SHA256,
        "experiment_summary_file":
            EXPERIMENT_SUMMARY_PATH.name,
        "experiment_summary_sha256":
            EXPERIMENT_SUMMARY_SHA256,
        "branch_C_structure_valid":
            bool(branch_c_structure_valid),
        "parsed_extraction_hash_matches_metadata":
            bool(parsed_extraction_hash_matches_metadata),
        "source_hash_matches_stage_1":
            bool(source_hash_matches_stage_1),
        "parent_B_equivalence_passed":
            bool(parent_equivalence_passed),
        "normalisation_integrity_passed":
            bool(normalisation_integrity_passed),
    },

    "comparison_rules_frozen_from_branch_A": True,
    "validation_timestamp":
        datetime.now(timezone.utc).isoformat(),
}

print(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "document_id": "D10",
  "document_name": "IMPI — Inquérito Mensal à Produção Industrial",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised layout-aware structural Markdown",
  "reference_records": 69,
  "extracted_records": 69,
  "aligned_records": 69,
  "fully_correct_records": 65,
  "discrepant_records": 4,
  "missing_records": 0,
  "unsupported_extracted_records": 0,
  "completeness": 1.0,
  "missing_rate": 0.0,
  "record_precision_exact": 0.9420289855072463,
  "record_recall_exact": 0.9420289855072463,
  "record_f1_exact": 0.9420289855072463,
  "unsupported_rate": 0.0,
  "discrepancy_rate_among_aligned": 0.057971014492753624,
  "overall_primary_field_accuracy": 0.9884057971014493,
  "primary_field_accuracy": {
    "Category": 1.0,
    "Field or Concept": 1.0,
    "Code": 0.9710144927536232,
    "Expected Value Type": 0.9710144927536232,
    "Source Location": 1.0
  },
  "diagnostic_field_accuracy":

In [13]:
# ============================================================
# 12. Export reproducible Validation C outputs
# ============================================================

SUMMARY_PATH = (
    OUTPUT_DIR / "D10_branch_C_validation_summary.json"
)

DETAILED_PATH = (
    OUTPUT_DIR / "D10_branch_C_validation_detailed.csv"
)

FULLY_CORRECT_PATH = (
    OUTPUT_DIR / "D10_branch_C_fully_correct_records.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR / "D10_branch_C_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR / "D10_branch_C_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR / "D10_branch_C_unsupported_records.csv"
)

FIELD_VALIDATION_PATH = (
    OUTPUT_DIR / "D10_branch_C_field_validation.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR / "D10_branch_C_category_metrics.csv"
)

REFERENCE_SEMANTICS_PATH = (
    OUTPUT_DIR / "D10_reference_semantics_confirmation.json"
)

ALIGNMENT_ISSUES_PATH = (
    OUTPUT_DIR / "D10_branch_C_alignment_issues.json"
)

VALIDATION_METADATA_PATH = (
    OUTPUT_DIR / "D10_branch_C_validation_metadata.json"
)

VALIDATION_CONCLUSION_PATH = (
    OUTPUT_DIR / "D10_branch_C_validation_conclusion.json"
)


fully_correct_records_df = (
    comparison_df.loc[
        comparison_df["Fully Correct Primary Record"]
    ].copy()
)

field_validation_df = pd.DataFrame([
    {
        "Field": field,
        "Role": (
            "Primary correctness"
            if field in PRIMARY_CORRECTNESS_FIELDS
            else "Diagnostic"
        ),
        "Correct": int(
            comparison_df[f"{field} Correct"].sum()
        ),
        "Compared": int(aligned_records),
        "Accuracy": float(
            comparison_df[f"{field} Correct"].mean()
        ) if aligned_records else 0.0,
    }
    for field in FIELDS
])

category_metrics_df = pd.DataFrame([
    {
        "Category": category,
        **metrics,
    }
    for category, metrics in category_metrics.items()
])


VALIDATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "validation_type":
        "Post-extraction reference-value agreement",
    "raw_extraction_modified": False,
    "manual_correction_applied": False,
    "comparison_normalisation_scope":
        "Comparison copies only",
    "comparison_rules_frozen_from_branch_A": True,
    "created_at":
        datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
}

validation_status = (
    "Completed without discrepancies"
    if (
        fully_correct_records == len(reference_df)
        and missing_records == 0
        and unsupported_records == 0
        and schema_validity
    )
    else "Completed with discrepancies"
)

VALIDATION_CONCLUSION = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "validation_status": validation_status,
    "reference_records": int(len(reference_df)),
    "extracted_records": int(len(extracted_df)),
    "aligned_records": int(aligned_records),
    "fully_correct_records": int(fully_correct_records),
    "discrepant_records": int(discrepant_records),
    "missing_records": int(missing_records),
    "unsupported_extracted_records": int(unsupported_records),
    "completeness": float(completeness),
    "record_precision_exact": float(record_precision_exact),
    "record_recall_exact": float(record_recall_exact),
    "record_f1_exact": float(record_f1_exact),
    "overall_primary_field_accuracy":
        float(overall_primary_field_accuracy),
    "schema_valid": bool(schema_validity),
    "normalisation_integrity_passed":
        bool(
            representation_integrity[
                "normalisation_integrity_passed"
            ]
        ),
    "parent_B_equivalence_passed":
        bool(
            representation_integrity[
                "parent_equivalence_passed"
            ]
        ),
    "comparison_rules_frozen_from_branch_A": True,
}


SUMMARY_PATH.write_text(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig",
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig",
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig",
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig",
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig",
)

REFERENCE_SEMANTICS_PATH.write_text(
    json.dumps(
        {
            "document_id": DOCUMENT_ID,
            "reference_semantics_valid":
                reference_semantics_valid,
            "checks":
                reference_semantic_checks,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

ALIGNMENT_ISSUES_PATH.write_text(
    json.dumps(
        ambiguous_alignment_groups,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

VALIDATION_METADATA_PATH.write_text(
    json.dumps(
        VALIDATION_METADATA,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

VALIDATION_CONCLUSION_PATH.write_text(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


assert reference_semantics_valid
assert branch_c_structure_valid
assert parsed_extraction_hash_matches_metadata
assert source_hash_matches_stage_1
assert representation_integrity["parent_equivalence_passed"]
assert representation_integrity["normalisation_integrity_passed"]
assert schema_validity

assert aligned_records + missing_records == len(reference_df)
assert aligned_records + unsupported_records == len(extracted_df)
assert fully_correct_records + discrepant_records == aligned_records

required_outputs = [
    SUMMARY_PATH,
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    FIELD_VALIDATION_PATH,
    CATEGORY_METRICS_PATH,
    REFERENCE_SEMANTICS_PATH,
    ALIGNMENT_ISSUES_PATH,
    VALIDATION_METADATA_PATH,
    VALIDATION_CONCLUSION_PATH,
]

missing_outputs = [
    path.name
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing output files: {missing_outputs}"
    )

print("Validation C — D10 completed successfully.")
print("Validation status:", validation_status)
print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_df))
print("Aligned records:", aligned_records)
print("Fully correct records:", fully_correct_records)
print("Discrepant records:", discrepant_records)
print("Missing records:", missing_records)
print("Unsupported/unmatched records:", unsupported_records)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Primary field accuracy:",
    round(overall_primary_field_accuracy, 4),
)
print("Schema valid:", schema_validity)
print(
    "Normalisation integrity passed:",
    representation_integrity[
        "normalisation_integrity_passed"
    ],
)
print(
    "Comparison rules frozen from Branch A:",
    True,
)

for path in required_outputs:
    print("-", path.name)

for path in required_outputs:
    files.download(path)

Validation C — D10 completed successfully.
Validation status: Completed with discrepancies
Reference records: 69
Extracted records: 69
Aligned records: 69
Fully correct records: 65
Discrepant records: 4
Missing records: 0
Unsupported/unmatched records: 0
Completeness: 1.0
Exact F1: 0.942
Primary field accuracy: 0.9884
Schema valid: True
Normalisation integrity passed: True
Comparison rules frozen from Branch A: True
- D10_branch_C_validation_summary.json
- D10_branch_C_validation_detailed.csv
- D10_branch_C_fully_correct_records.csv
- D10_branch_C_discrepant_records.csv
- D10_branch_C_missing_records.csv
- D10_branch_C_unsupported_records.csv
- D10_branch_C_field_validation.csv
- D10_branch_C_category_metrics.csv
- D10_reference_semantics_confirmation.json
- D10_branch_C_alignment_issues.json
- D10_branch_C_validation_metadata.json
- D10_branch_C_validation_conclusion.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>